## 0. Setup (run every session)
Check the GPU type below matches what this job needs, then clone + install.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('/content/COCO-Urdu'):
    !git clone -q -b v2 https://github.com/umair-hassan2/COCO-Urdu.git /content/COCO-Urdu
%cd /content/COCO-Urdu
!git pull -q
!pip install -q -e .

In [ ]:
import os
# Token source, in order: an HF_TOKEN already in the environment, else Colab Secrets
# (sidebar key icon -> add HF_TOKEN, enable for this notebook). Needs WRITE scope: every job
# pushes its outputs to the private artifacts repo via scripts/sync.py.
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
except Exception:
    pass
assert os.environ.get('HF_TOKEN'), 'No HF_TOKEN. Add it in the Colab sidebar (key icon) as HF_TOKEN.'

# huggingface_hub reads HF_TOKEN from the environment, and the `!python` cells below
# are subprocesses that inherit it -- so no notebook_login() prompt is needed.
from huggingface_hub import whoami
_me = whoami(token=os.environ['HF_TOKEN'])
print('HF auth OK:', _me['name'], '| token role:', _me.get('auth', {}).get('accessToken', {}).get('role'))


## 3. xCOMET-XL on the human-study items — **L4**, ~2 units (skip if your Mac has ≥32 GB RAM)
Needs `data/study/items.jsonl`, pushed from the Mac after step 5: `python scripts/sync.py push data/study/items.jsonl`.

In [ ]:
!python scripts/sync.py pull data/study/items.jsonl
# if the Mac already scored comet_kiwi/metricx/siglip2, pull that table so xcomet
# is added as a column instead of starting a fresh one
!python scripts/sync.py pull --if-present data/study/qe_scores.parquet


In [ ]:
!python scripts/06_score_qe.py --metrics xcomet

### Publish the outputs
On the Mac: `python scripts/sync.py pull data/study/qe_scores.parquet`.

In [ ]:
!python scripts/sync.py push data/study/qe_scores.parquet
